# 11. 接收机基础

**Receiver Basics**

---

本notebook是通信基础系列的第十一课，将帮助您理解接收机的基本原理和关键技术。接收机是通信系统中至关重要的一环，负责从收到的信号中恢复出发送的原始信息。


## 1. 学习目标

通过本notebook，您将：

- **理解匹配滤波器的原理**：掌握如何在噪声环境下最大化信号接收质量
- **理解载波同步**：掌握Costas环的工作原理及其在载波恢复中的作用
- **理解符号同步**：掌握Early-Late门的原理和实现
- **理解基本的均衡概念**：了解线性均衡器和判决反馈均衡器的区别
- **理解眼图的作用**：掌握眼图如何用于评估符号间干扰

## 2. 概念讲解

### 2.1 匹配滤波器

匹配滤波器（Matched Filter）是一种最优线性滤波器，其设计目标是**最大化输出信噪比（SNR）**。在数字通信中，匹配滤波器与发送的信号形状匹配，能够在噪声存在的情况下最佳地恢复发送的符号。

**核心原理**：
如果发送的信号为$s(t)$，则匹配滤波器的冲激响应为：

$$h(t) = s(T-t)$$

其中$T$为符号周期。匹配滤波器的输出在$t=T$时刻采样，得到最大SNR。


### 2.2 载波同步

载波同步（Carrier Synchronization）是接收机恢复发射机载波过程的技术。由于信道延迟和多普勒效应，接收信号的载波相位会与本地振荡器存在偏差。

**Costas环**

Costas环是一种闭环载波同步算法，广泛应用于BPSK和QPSK系统的载波恢复。其基本思想是利用鉴相器检测接收信号与本地载波之间的相位差，并通过环路滤波器控制VCO（压控振荡器）跟踪真实相位。

Costas环的优点：
- 可以处理相位模糊（0°、90°、180°、270°）
- 适用于低信噪比环境
- 硬件实现相对简单


### 2.3 符号同步

符号同步（Symbol Synchronization）用于确定最佳的采样时刻。接收机需要知道每个符号周期的最佳采样点，以最大化信号幅度并最小化符号间干扰。

**Early-Late门同步算法**

Early-Late门是一种基于能量的符号同步算法。原理如下：
- 在估计的符号周期附近，在**早**（Early）和**晚**（Late）两个时刻进行采样
- 比较两个采样值的大小：
  - 如果$E > L$，说明采样时刻偏早，应提前
  - 如果$E < L$，说明采样时刻偏晚，应推迟
  - 如果$E = L$，采样时刻最佳
- 根据比较结果调整采样时刻


### 2.4 均衡器

由于多径传播，接收信号会经历频率选择性衰落，导致符号间干扰（ISI）。均衡器（Equalizer）用于补偿信道引起的失真，恢复发送的原始数据。

**线性均衡器（LE）**

线性均衡器采用横向滤波器结构，其输出是过去若干个符号的线性组合：

$$y[n] = \sum_{k=-N}^{N} w_k \cdot x[n-k]$$

其中$w_k$为均衡器系数，$N$为滤波器阶数。

**判决反馈均衡器（DFE）**

判决反馈均衡器由前向滤波器和反馈滤波器组成。反馈滤波器利用已判决的符号来消除当前符号的ISI：

$$y[n] = \sum_{k=0}^{N_1} w_k \cdot x[n-k] - \sum_{k=1}^{N_2} b_k \cdot \hat{y}[n-k]$$

DFE在非线性信道（如高频衰落信道）中性能优于线性均衡器。


### 2.5 眼图

眼图（Eye Diagram）是评估数字通信系统性能的重要工具。将接收信号的多个符号周期重叠显示，由于符号间干扰和噪声的影响，眼图的"眼睛"会张开或闭合。

**眼图的关键指标**：
- **眼开度**：反映符号间干扰的程度，眼开度越大，系统性能越好
- **最佳采样点**：眼图张开最大的位置，对应最佳采样时刻
- **过零点偏移**：反映定时误差的大小
- **噪声边际**：接收机对噪声的容限


## 3. Python演示

下面我们使用Python来演示接收机的各项关键技术。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import lfilter, firwin
%matplotlib inline

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("接收机基础演示环境已准备就绪")

### 3.1 匹配滤波器演示


In [ ]:
def matched_filter_demo():
    """
    演示匹配滤波器的工作原理
    
    对比匹配滤波器与普通低通滤波器的输出信噪比
    """
    # 系统参数
    symbol_rate = 1e6       # 符号率 1 MHz
    samples_per_symbol = 32  # 每符号采样点数
    num_symbols = 100       # 符号数量
    snr_db = 10             # 信噪比(dB)
    
    # 简化的矩形脉冲
    pulse = np.ones(samples_per_symbol)
    pulse = pulse / np.sqrt(np.sum(pulse**2))  # 归一化
    
    # 生成随机比特
    bits = np.random.randint(0, 2, num_symbols)
    symbols = 2 * bits - 1  # BPSK符号: +1 或 -1
    
    # 生成发送信号
    tx_signal = np.zeros(num_symbols * samples_per_symbol)
    for i, sym in enumerate(symbols):
        tx_signal[i*samples_per_symbol:(i+1)*samples_per_symbol] = sym * pulse
    
    # 添加噪声
    signal_power = np.mean(tx_signal**2)
    noise_std = np.sqrt(signal_power / (10**(snr_db/10)))
    rx_signal = tx_signal + noise_std * np.random.randn(len(tx_signal))
    
    # 匹配滤波器（与脉冲匹配）
    matched_filter_output = np.convolve(rx_signal, pulse[::-1], mode='same')
    
    # 普通低通滤波器（对比）
    lpf = firwin(64, 0.3, window='hamming')  # 截止频率0.3（归一化）
    lpf_output = np.convolve(rx_signal, lpf, mode='same')
    
    # 采样判决（每symbols_per_symbol点采样一次）
    mid_points = np.arange(samples_per_symbol//2, len(matched_filter_output), samples_per_symbol)
    matched_sampled = matched_filter_output[mid_points]
    lpf_sampled = lpf_output[mid_points]
    
    # 计算信噪比增益
    mf_signal = np.mean(matched_sampled[:len(bits)] ** 2)
    mf_noise = np.var(matched_sampled[:len(bits)] - bits * np.sign(matched_sampled[:len(bits)]))
    mf_snr = 10 * np.log10(mf_signal / (mf_noise + 1e-10))
    
    lpf_signal = np.mean(lpf_sampled[:len(bits)] ** 2)
    lpf_noise = np.var(lpf_sampled[:len(bits)] - bits * np.sign(lpf_sampled[:len(bits)]))
    lpf_snr = 10 * np.log10(lpf_signal / (lpf_noise + 1e-10))
    
    # 绘制结果
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    
    # 时域信号对比
    time = np.arange(200) / samples_per_symbol
    axes[0, 0].plot(time, rx_signal[:200], 'b-', alpha=0.7, label='接收信号(带噪声)')
    axes[0, 0].plot(time, tx_signal[:200], 'g--', alpha=0.7, label='发送信号')
    axes[0, 0].set_xlabel('时间 (符号周期)')
    axes[0, 0].set_ylabel('幅度')
    axes[0, 0].set_title('接收信号与发送信号对比')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 匹配滤波器输出
    axes[0, 1].plot(time, matched_filter_output[:200], 'b-', label='匹配滤波器输出')
    axes[0, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    for i in range(5):
        axes[0, 1].axvline(x=(i + 0.5), color='r', linestyle=':', alpha=0.5)
    axes[0, 1].set_xlabel('时间 (符号周期)')
    axes[0, 1].set_ylabel('幅度')
    axes[0, 1].set_title('匹配滤波器输出')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 低通滤波器输出
    axes[1, 0].plot(time, lpf_output[:200], 'orange', label='低通滤波器输出')
    axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    axes[1, 0].set_xlabel('时间 (符号周期)')
    axes[1, 0].set_ylabel('幅度')
    axes[1, 0].set_title('普通低通滤波器输出')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 采样判决效果对比
    decision_point = np.arange(20)
    axes[1, 1].stem(decision_point, matched_sampled[:20], linefmt='b-', markerfmt='bo', basefmt='k-', label='匹配滤波器')
    axes[1, 1].stem([x + 0.3 for x in decision_point], lpf_sampled[:20], linefmt='orange', markerfmt='s', basefmt='k-', label='低通滤波器')
    axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    axes[1, 1].set_xlabel('符号序号')
    axes[1, 1].set_ylabel('采样值')
    axes[1, 1].set_title('采样判决点对比')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # SNR对比条形图
    axes[2, 0].bar(['匹配滤波器', '低通滤波器'], [mf_snr, lpf_snr], color=['blue', 'orange'])
    axes[2, 0].set_ylabel('输出SNR (dB)')
    axes[2, 0].set_title('滤波器输出信噪比对比')
    for i, v in enumerate([mf_snr, lpf_snr]):
        axes[2, 0].text(i, v + 0.5, f'{v:.1f} dB', ha='center', fontsize=10)
    axes[2, 0].grid(True, alpha=0.3, axis='y')
    
    # 星座图对比
    axes[2, 1].scatter(matched_sampled[:100], np.zeros(100), alpha=0.5, s=30, label='匹配滤波器')
    axes[2, 1].scatter(lpf_sampled[:100], np.zeros(100), alpha=0.5, s=30, label='低通滤波器')
    axes[2, 1].axvline(x=0, color='k', linestyle='--', alpha=0.3)
    axes[2, 1].set_xlabel('采样值')
    axes[2, 1].set_ylabel('')
    axes[2, 1].set_title('判决空间分布')
    axes[2, 1].legend()
    axes[2, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n匹配滤波器输出SNR: {mf_snr:.2f} dB")
    print(f"低通滤波器输出SNR: {lpf_snr:.2f} dB")
    print(f"匹配滤波器增益: {mf_snr - lpf_snr:.2f} dB")
    
matched_filter_demo()

### 3.2 眼图生成演示


In [ ]:
def eye_diagram_demo():
    """
    演示眼图的生成和作用
    
    展示不同条件下的眼图变化：无噪声、有噪声、不同时序误差
    """
    # 系统参数
    samples_per_symbol = 64
    num_symbols = 500
    
    # 生成BPSK信号
    bits = np.random.randint(0, 2, num_symbols)
    symbols = 2 * bits - 1
    
    # 矩形脉冲
    pulse = np.ones(samples_per_symbol)
    
    # 生成基带信号
    tx_signal = np.repeat(symbols, samples_per_symbol) * np.tile(pulse, num_symbols)
    
    # 1. 无噪声理想情况
    ideal_signal = tx_signal.copy()
    
    # 2. 添加噪声
    noise_std = 0.3
    noisy_signal = ideal_signal + noise_std * np.random.randn(len(ideal_signal))
    
    # 3. 多径衰落（模拟符号间干扰）
    multipath = np.convolve(noisy_signal, [1, 0.3, 0.1], mode='same')[:len(noisy_signal)]
    
    # 绘制眼图
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    def plot_eye(signal_data, ax, title, samples_per_symbol=64, num_traces=20):
        """绘制眼图"""
        traces = []
        for i in range(num_traces):
            start = i * samples_per_symbol * 3  # 每隔3个符号取一段
            if start + samples_per_symbol * 2 <= len(signal_data):
                traces.append(signal_data[start:start + samples_per_symbol * 2])
        
        traces = np.array(traces)
        time_axis = np.arange(traces.shape[1]) / samples_per_symbol
        
        for trace in traces:
            ax.plot(time_axis, trace, 'b-', alpha=0.3, linewidth=0.5)
        
        # 计算平均眼图
        mean_trace = np.mean(traces, axis=0)
        ax.plot(time_axis, mean_trace, 'r-', linewidth=2, label='平均值')
        
        ax.set_xlabel('时间 (符号周期)')
        ax.set_ylabel('幅度')
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 2])
        
        # 标注最佳采样点
        ax.axvline(x=1, color='g', linestyle='--', alpha=0.7, label='最佳采样点')
        ax.legend(loc='upper right')
    
    plot_eye(ideal_signal, axes[0, 0], '眼图（理想无噪声）')
    plot_eye(noisy_signal, axes[0, 1], '眼图（加性高斯白噪声）')
    plot_eye(multipath, axes[1, 0], '眼图（多径衰落 - 符号间干扰）')
    
    # 4. 定时误差的信号
    timing_error_signal = np.zeros_like(noisy_signal)
    for i in range(0, len(noisy_signal) - 1, samples_per_symbol):
        offset = int((i // samples_per_symbol) % 3 == 0)  # 每3个符号偏移1点
        if i + offset < len(timing_error_signal):
            timing_error_signal[i] = noisy_signal[i]
            timing_error_signal[i+offset:i+samples_per_symbol] = noisy_signal[i+1:i+samples_per_symbol+1-offset][:samples_per_symbol-offset]
    
    plot_eye(timing_error_signal, axes[1, 1], '眼图（定时误差）')
    
    plt.tight_layout()
    plt.show()
    
    # 分析眼图指标
    print("\n眼图分析：")
    print("=" * 50)
    print("1. 理想眼图：眼睛完全张开，无噪声，无符号间干扰")
    print("2. 加噪眼图：眼睛仍然张开，但有噪声导致的抖动")
    print("3. 多径眼图：眼睛闭合程度增大，符号间干扰严重")
    print("4. 定时误差眼图：最佳采样点偏移，系统性能下降")
    print("\n关键观察：")
    print("- 眼开度越大，系统对噪声和定时误差的容忍度越高")
    print("- 符号间干扰会"闭合"眼图，导致误码率增加")
    
eye_diagram_demo()

### 3.3 均衡器性能对比


In [ ]:
def equalizer_comparison_demo():
    """
    对比线性均衡器和判决反馈均衡器的性能
    
    模拟多径信道，比较两种均衡器的输出质量
    """
    # 系统参数
    num_symbols = 1000
    samples_per_symbol = 16
    snr_db = 15
    
    # 生成随机数据
    bits = np.random.randint(0, 2, num_symbols)
    tx_symbols = 2 * bits - 1  # BPSK
    
    # 多径信道 [直接路径, 反射路径1, 反射路径2]
    channel = np.array([1.0, 0.3, 0.15])  # 多径增益
    channel_delays = np.array([0, 1, 2])   # 延迟（采样点）
    
    # 创建多径信号
    def create_multipath(symbols, channel, delays, sps):
        """创建多径信号"""
        output = np.zeros(len(symbols) * sps + max(delays) * sps)
        for i, (gain, delay) in enumerate(zip(channel, delays)):
            for j, sym in enumerate(symbols):
                start = j * sps + delay * sps
                output[start:start + sps] += sym * gain * np.ones(sps)
        return output[:num_symbols * sps]
    
    # 生成发射信号
    tx_signal = np.repeat(tx_symbols, samples_per_symbol)
    
    # 通过多径信道
    rx_multipath = create_multipath(tx_symbols, channel, channel_delays, samples_per_symbol)
    
    # 添加噪声
    signal_power = np.mean(rx_multipath**2)
    noise_std = np.sqrt(signal_power / (10**(snr_db/10)))
    rx_signal = rx_multipath + noise_std * np.random.randn(len(rx_multipath))
    
    # 下采样到符号速率
    rx_symbols = rx_signal[::samples_per_symbol]
    
    # 线性均衡器（迫零算法）
    # 信道估计
    channel_estimated = np.convolve(rx_symbols, np.sign(tx_symbols), mode='full')
    channel_estimated = channel_estimated[len(tx_symbols)-1:len(tx_symbols)+2]  # 取主要部分
    channel_estimated = channel_estimated / np.max(np.abs(channel_estimated))
    
    # 迫零均衡器
    le_coef = 1.0 / (channel_estimated + 1e-6)
    le_output = rx_symbols * le_coef
    
    # 判决反馈均衡器
    dfe_forward = 1.0 / (channel_estimated[0] + 1e-6)  # 前向系数
    dfe_feedback = -channel_estimated[1:] / (channel_estimated[0] + 1e-6)  # 反馈系数
    
    dfe_output = np.zeros_like(rx_symbols)
    detected_symbols = np.zeros_like(rx_symbols)
    
    for i in range(len(rx_symbols)):
        # 前向滤波
        forward_part = dfe_forward * rx_symbols[i]
        # 反馈滤波（使用已判决符号）
        feedback_part = 0
        for j, coef in enumerate(dfe_feedback):
            if i - j - 1 >= 0:
                feedback_part += coef * detected_symbols[i - j - 1]
        
        dfe_output[i] = forward_part - feedback_part
        # 判决
        detected_symbols[i] = np.sign(dfe_output[i])
        dfe_output[i] = detected_symbols[i]  # 记录判决后的值
    
    # 计算误码率
    le_errors = np.sum(np.abs((le_output > 0).astype(int) - bits) > 0)
    dfe_errors = np.sum(np.abs((dfe_output > 0).astype(int) - bits) > 0)
    no_eq_errors = np.sum(np.abs((rx_symbols > 0).astype(int) - bits) > 0)
    
    le_ber = le_errors / num_symbols
    dfe_ber = dfe_errors / num_symbols
    no_eq_ber = no_eq_errors / num_symbols
    
    # 绘制结果
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 星座图对比
    axes[0, 0].scatter(rx_symbols[::4], np.zeros(len(rx_symbols[::4])), alpha=0.3, s=10, c='gray')
    axes[0, 0].set_title('无均衡器（直接判决）')
    axes[0, 0].set_xlabel('幅度')
    axes[0, 0].set_ylabel('')
    axes[0, 0].axvline(x=0, color='k', linestyle='--', alpha=0.3)
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].scatter(le_output[::4], np.zeros(len(le_output[::4])), alpha=0.3, s=10, c='blue')
    axes[0, 1].set_title('线性均衡器 (LE)')
    axes[0, 1].set_xlabel('幅度')
    axes[0, 1].set_ylabel('')
    axes[0, 1].axvline(x=0, color='k', linestyle='--', alpha=0.3)
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].scatter(dfe_output[::4], np.zeros(len(dfe_output[::4])), alpha=0.3, s=10, c='green')
    axes[1, 0].set_title('判决反馈均衡器 (DFE)')
    axes[1, 0].set_xlabel('幅度')
    axes[1, 0].set_ylabel('')
    axes[1, 0].axvline(x=0, color='k', linestyle='--', alpha=0.3)
    axes[1, 0].grid(True, alpha=0.3)
    
    # BER对比条形图
    methods = ['无均衡', '线性均衡', '判决反馈均衡']
    bers = [no_eq_ber, le_ber, dfe_ber]
    colors = ['gray', 'blue', 'green']
    bars = axes[1, 1].bar(methods, bers, color=colors)
    axes[1, 1].set_ylabel('误码率 (BER)')
    axes[1, 1].set_title('误码率性能对比')
    axes[1, 1].set_yscale('log')
    
    for bar, ber in zip(bars, bers):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height * 1.2,
                       f'{ber:.4f}', ha='center', va='bottom', fontsize=10)
    
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n多径信道: {channel}")
    print(f"延迟: {channel_delays} 采样点")
    print(f"\n误码率对比：")
    print(f"  无均衡器:   {no_eq_ber:.4f} ({no_eq_errors}/{num_symbols})")
    print(f"  线性均衡器: {le_ber:.4f} ({le_errors}/{num_symbols})")
    print(f"  判决反馈均衡器: {dfe_ber:.4f} ({dfe_errors}/{num_symbols})")
    
equalizer_comparison_demo()

### 3.4 Early-Late门符号同步演示


In [ ]:
def early_late_gate_demo():
    """
    演示Early-Late门符号同步算法
    
    展示同步环如何跟踪最佳采样时刻
    """
    # 参数设置
    samples_per_symbol = 32
    num_symbols = 200
    snr_db = 10
    
    # 生成数据
    bits = np.random.randint(0, 2, num_symbols)
    tx_symbols = 2 * bits - 1
    
    # 矩形脉冲
    pulse = np.ones(samples_per_symbol)
    
    # 生成信号
    tx_signal = np.repeat(tx_symbols, samples_per_symbol)
    
    # 添加噪声
    signal_power = np.mean(tx_signal**2)
    noise_std = np.sqrt(signal_power / (10**(snr_db/10)))
    rx_signal = tx_signal + noise_std * np.random.randn(len(tx_signal))
    
    # Early-Late门同步参数
    delta = 1  # Early/Late间隔（采样点）
    alpha = 0.01  # 步长
    
    # 初始采样位置
    tau = samples_per_symbol // 2  # 初始估计：符号中心
    
    # 存储同步过程
    tau_history = [tau]
    e_minus_l_history = []
    
    # 匹配滤波器
    matched_filter = pulse[::-1]
    mf_output = np.convolve(rx_signal, matched_filter, mode='same')
    
    # Early-Late门同步环
    for i in range(num_symbols - 1):
        # Early采样（提前）
        early_idx = int(i * samples_per_symbol + tau - delta)
        late_idx = int(i * samples_per_symbol + tau + delta)
        
        if early_idx >= 0 and early_idx < len(mf_output) and late_idx >= 0 and late_idx < len(mf_output):
            E = np.abs(mf_output[early_idx])  # Early能量
            L = np.abs(mf_output[late_idx])   # Late能量
            
            # 误差信号：E - L
            error = E - L
            e_minus_l_history.append(error)
            
            # 更新采样时刻
            tau = tau - alpha * error
            
            # 限制tau的范围
            tau = np.clip(tau, samples_per_symbol // 4, 3 * samples_per_symbol // 4)
            
            tau_history.append(tau)
    
    # 计算最终采样点
    final_samples = []
    for i in range(num_symbols):
        idx = int(i * samples_per_symbol + tau_history[-1])
        if idx >= 0 and idx < len(mf_output):
            final_samples.append(mf_output[idx])
    final_samples = np.array(final_samples)
    
    # 绘制结果
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 同步过程
    time_sync = np.arange(len(tau_history)) / num_symbols
    axes[0, 0].plot(time_sync, tau_history, 'b-', linewidth=1)
    axes[0, 0].axhline(y=samples_per_symbol // 2, color='r', linestyle='--', label='理想采样点')
    axes[0, 0].set_xlabel('时间（符号数）')
    axes[0, 0].set_ylabel('采样时刻 (tau)')
    axes[0, 0].set_title('Early-Late门同步：采样时刻跟踪')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 误差信号
    axes[0, 1].plot(e_minus_l_history, 'g-', alpha=0.5, linewidth=0.5)
    axes[0, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    axes[0, 1].set_xlabel('迭代次数')
    axes[0, 1].set_ylabel('E - L')
    axes[0, 1].set_title('误差信号 (E-L) 变化')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 采样点分布
    axes[1, 0].scatter(final_samples[:50], np.zeros(50), alpha=0.5, s=30)
    axes[1, 0].axvline(x=0, color='k', linestyle='--', alpha=0.3)
    axes[1, 0].set_xlabel('采样值')
    axes[1, 0].set_ylabel('')
    axes[1, 0].set_title('同步后的采样点分布')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 原始信号与同步采样
    time = np.arange(4 * samples_per_symbol) / samples_per_symbol
    axes[1, 1].plot(time, rx_signal[:4*samples_per_symbol], 'b-', alpha=0.7, label='接收信号')
    
    # 标注采样点
    sample_positions = [int(tau_history[-1]) + i * samples_per_symbol for i in range(5)]
    for pos in sample_positions[:4]:
        if pos < len(rx_signal):
            axes[1, 1].axvline(x=pos/samples_per_symbol, color='g', linestyle=':', alpha=0.7)
    
    axes[1, 1].set_xlabel('时间 (符号周期)')
    axes[1, 1].set_ylabel('幅度')
    axes[1, 1].set_title('采样点位置示意（绿色虚线）')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 计算同步误差
    ideal_tau = samples_per_symbol // 2
    final_error = np.abs(tau_history[-1] - ideal_tau)
    
    print(f"\nEarly-Late门同步结果：")
    print(f"  初始采样时刻: {tau_history[0]}")
    print(f"  最终采样时刻: {tau_history[-1]:.2f}")
    print(f"  理想采样时刻: {ideal_tau}")
    print(f"  同步误差: {final_error:.2f} 采样点")
    print(f"\n原理说明：")
    print("  当E > L时，采样点偏早，应提前（tau减小）")
    print("  当E < L时，采样点偏晚，应推迟（tau增大）")
    print("  当E = L时，采样点在最佳位置")
    
early_late_gate_demo()

## 4. 原理解析

### 4.1 同步误差对系统性能的影响

同步误差（包括载波相位误差和定时误差）会显著降低通信系统的性能。

**载波相位误差的影响**：

对于BPSK系统，载波相位误差$\Delta\phi$会导致信号能量损失：

$$E_{loss} = 20 \log_{10}(\cos(\Delta\phi)) \quad [dB]$$

当$\Delta\phi = 30°$时，损失约1.25 dB；$\Delta\phi = 90°$时，完全丢失信号。

**定时误差的影响**：

定时误差导致采样点偏移最佳位置，引入符号间干扰。眼图的"眼睛"会随着定时误差增大而闭合。

**综合影响**：

同步误差与噪声共同作用，会使系统误码率急剧上升。


In [ ]:
def sync_error_effect_demo():
    """
    演示同步误差对系统性能的影响
    """
    # 参数设置
    num_symbols = 5000
    snr_db_range = np.arange(5, 15, 2)
    
    # 相位误差范围（度）
    phase_errors = np.array([0, 10, 30, 60, 90])
    
    # 定时误差范围（采样点，相对符号周期）
    timing_errors = np.array([0, 0.05, 0.1, 0.2, 0.3])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. 相位误差对BER的影响
    print("计算相位误差对误码率的影响...")
    
    for phase_err in phase_errors:
        bers = []
        for snr_db in snr_db_range:
            # 生成BPSK信号
            bits = np.random.randint(0, 2, num_symbols)
            symbols = 2 * bits - 1
            
            # 添加噪声
            signal_power = 1.0
            noise_std = np.sqrt(signal_power / (10**(snr_db/10)))
            rx_symbols = symbols + noise_std * np.random.randn(num_symbols)
            
            # 加入相位误差
            phase_rad = np.deg2rad(phase_err)
            rx_symbols = rx_symbols * np.cos(phase_rad) + np.zeros(num_symbols) * np.sin(phase_rad)
            
            # 判决计算BER
            detected = np.sign(rx_symbols)
            errors = np.sum(detected != symbols)
            bers.append(errors / num_symbols)
        
        label = f'相位误差 = {phase_err}' if phase_err > 0 else '无相位误差'
        axes[0].semilogy(snr_db_range, bers, 'o-', label=label)
    
    axes[0].set_xlabel('SNR (dB)')
    axes[0].set_ylabel('BER')
    axes[0].set_title('载波相位误差对BER的影响')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([1e-4, 1])
    
    # 2. 定时误差对BER的影响
    print("计算定时误差对误码率的影响...")
    
    samples_per_symbol = 32
    for timing_err in timing_errors:
        bers = []
        for snr_db in snr_db_range:
            # 生成信号
            bits = np.random.randint(0, 2, num_symbols)
            symbols = 2 * bits - 1
            
            # 创建矩形脉冲信号
            tx_signal = np.repeat(symbols, samples_per_symbol)
            pulse = np.ones(samples_per_symbol)
            tx_signal = np.convolve(tx_signal, pulse, mode='same')[:num_symbols * samples_per_symbol]
            tx_signal = tx_signal[::samples_per_symbol]  # 下采样回符号率
            
            # 添加噪声
            signal_power = np.mean(tx_signal**2)
            noise_std = np.sqrt(signal_power / (10**(snr_db/10)))
            rx_symbols = tx_signal + noise_std * np.random.randn(num_symbols)
            
            # 在最佳采样点加入定时误差
            if timing_err > 0:
                # 简单模型：定时误差导致判决点偏移
                rx_symbols = rx_symbols * (1 - timing_err * np.abs(rx_symbols) / signal_power)
            
            # 判决
            detected = np.sign(rx_symbols)
            errors = np.sum(detected != symbols)
            bers.append(errors / num_symbols)
        
        label = f'定时误差 = {timing_err}' if timing_err > 0 else '无定时误差'
        axes[1].semilogy(snr_db_range, bers, 's-', label=label)
    
    axes[1].set_xlabel('SNR (dB)')
    axes[1].set_ylabel('BER')
    axes[1].set_title('定时误差对BER的影响')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([1e-4, 1])
    
    plt.tight_layout()
    plt.show()
    
    print("\n分析结论：")
    print("- 相位误差会直接导致信号能量损失，显著增加误码率")
    print("- 定时误差导致采样点偏离最佳位置，引入额外噪声")
    print("- 两种误差在高斯噪声背景下影响更严重")
    
sync_error_effect_demo()

## 5. 思考题

1. **匹配滤波器**
   - 匹配滤波器的核心优势是什么？为什么它能够最大化输出信噪比？
   - 如果发送脉冲不是矩形脉冲，匹配滤波器的设计会有何变化？

2. **载波同步**
   - Costas环如何解决相位模糊问题（0、90、180、270度）？
   - 在高动态场景（如高速移动）下，Costas环可能面临什么挑战？

3. **符号同步**
   - Early-Late门算法中，步长alpha的选择有什么讲究？
   - 除了Early-Late门外，还有哪些符号同步算法？

4. **均衡器**
   - 为什么判决反馈均衡器（DFE）在非线性信道中表现更好？
   - 线性均衡器和DFE各自的优缺点是什么？

5. **眼图**
   - 通过眼图，如何判断一个通信系统的性能优劣？
   - 如果眼图完全闭合，说明系统存在什么问题？

---

## 总结

本notebook介绍了接收机的基本原理和关键技术：

- **匹配滤波器**：在噪声环境下最大化信号质量，是数字接收机的核心组件
- **载波同步**：通过Costas环等算法恢复发射机载波，消除相位偏移
- **符号同步**：通过Early-Late门等算法确定最佳采样时刻
- **均衡器**：补偿多径衰落引起的符号间干扰
- **眼图**：评估系统性能的综合性工具

这些技术共同构成了现代通信系统接收机的基础，使得信息能够在噪声和干扰存在的情况下可靠传输。